In [ ]:
!git clone https://github.com/RajvilChoudhary/PeerReview.git

Cloning into 'PeerReview'...
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 8 (delta 1), reused 8 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (8/8), 11.06 KiB | 11.06 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [ ]:
# Change directory to your cloned repository folder
# (Replace 'peerreview' with your exact repository folder name if it's different)
%cd /content/PeerReview

# Check that autoassigner.py is listed here
!ls

/content/PeerReview
peerReview.ipynb  peerReviewOptimized.ipynb  peerReviewSol2.ipynb  README.md


In [ ]:
!pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 55.5 MB/s eta 0:00:00


In [6]:
#!/usr/bin/env python3
import numpy as np

from gurobipy import Model, GRB, quicksum
import autoassigner  # Core peer-review assignment engine containing the PR4A class

def generate_dataset(n_reviewers=30, n_papers=20, lambda_=3, mu_=5):
    """
    Generates a synthetic matrix simulating reviewer-to-paper similarity scores.

    To properly test assignment algorithms, this function deliberately creates a
    "bottleneck" or "niche" paper (at index 0). This paper has very low similarity
    scores with almost all reviewers, making it a critical test case for Max-Min Fairness.

    Parameters:
    -----------
    n_reviewers : int, Total number of available reviewers in the pool.
    n_papers    : int, Total number of submitted papers requiring review.
    lambda_     : int, Number of reviewers required per paper (demand).
    mu_         : int, Maximum number of papers a single reviewer can handle (capacity).

    Returns:
    --------
    S : np.ndarray, A 2D matrix of shape (n_reviewers, n_papers) where entry S[i, j]
                    represents the continuous similarity score between reviewer i and paper j.
    """
    # Using a fixed random seed ensures the dataset is identical across runs for fair benchmarking
    np.random.seed(98)

    # Generate baseline random similarities uniformly distributed between 0.05 and 0.95
    S = np.random.uniform(0.05, 0.95, (n_reviewers, n_papers))

    # --- CONSTRUCT THE BOTTLENECK TEST CASE ---
    # Force Paper 0 to have low similarity (0.05 to 0.20) across the entire pool...
    S[:, 0] = np.random.uniform(0.05, 0.2, n_reviewers)
    # ...except for exactly two reviewers who are excellent matches.
    # Since lambda_ (demand) is usually >= 3, this creates an unavoidable deficit
    # where Paper 0 is guaranteed to get at least one low-quality reviewer.
    S[0, 0] = 0.90
    S[1, 0] = 0.85

    return S

def run_binary_nsw(S, lambda_, mu_, threshold=0.4):
    """
    Solves the peer assignment problem using the Binary Nash Social Welfare (NSW) framework.

    Instead of optimizing continuous scores directly, this algorithm simplifies the
    problem by treating matches as binary: "Good" (above threshold) or "Bad" (below threshold).
    It then maximizes global welfare using a logarithmic utility function, which naturally
    balances total high-quality matches while preventing any single paper from being neglected.

    Mathematical Note:
    ------------------
    Nash Social Welfare maximizes the product of utilities: Objective = MultiProduct(Utility_j).
    Taking the logarithm turns this into a linear sum: Objective = Sum( log(Utility_j) ).
    This function uses binary integer programming to solve this log-sum objective.
    """
    n, m = S.shape

    # Step 1: Convert continuous similarity matrix S into a binary matrix B.
    # B[i, j] = 1 if reviewer i is a "good match" for paper j, otherwise 0.
    B = (S >= threshold).astype(int)

    # Step 2: Initialize the Gurobi Optimization Model
    model = Model("Binary_NSW")
    model.setParam('OutputFlag', False) # Suppress detailed optimization logs for clean console output

    # Step 3: Define Decision Variables
    # x[i, j] is a binary variable: 1 if reviewer i is assigned to paper j, 0 otherwise.
    x = model.addVars(n, m, vtype=GRB.BINARY, name="assign")

    # Step 4: Define Constraints
    # Constraint A: Do not overload reviewers. Total papers assigned to reviewer i cannot exceed mu_.
    model.addConstrs(x.sum(i, '*') <= mu_ for i in range(n))
    # Constraint B: Satisfy paper demands. Total reviewers assigned to paper j must exactly equal lambda_.
    model.addConstrs(x.sum('*', j) == lambda_ for j in range(m))

    # Step 5: Linearize the Logarithmic Objective Function
    # Gurobi handles linear/quadratic functions best. To model the non-linear log function,
    # we introduce indicator variables w[j, k].
    # w[j, k] = 1 if paper j receives AT LEAST k "good" reviewers (where B[i, j] == 1).
    w = model.addVars(m, lambda_ + 1, vtype=GRB.BINARY, name="w")

    for j in range(m):
        # Count how many assigned reviewers for paper j qualify as "good matches"
        good_count = quicksum(x[i, j] for i in range(n) if B[i, j] == 1)

        # Link the count to our indicator variables. If good_count = 2, then w[j,1]=1 and w[j,2]=1.
        model.addConstr(good_count >= quicksum(w[j, k] for k in range(1, lambda_ + 1)))

    # Step 6: Define the Objective Function
    # The marginal utility of getting the k-th good reviewer is modeled as: log(k + 1) - log(k).
    # This captures diminishing returns (the 1st good reviewer adds massive value, the 3rd adds less).
    # It also uses log(k+1) rather than log(k) to prevent math errors when a paper gets 0 good reviewers.
    obj = quicksum((np.log(k + 1) - np.log(k)) * w[j, k] for j in range(m) for k in range(1, lambda_ + 1))
    model.setObjective(obj, GRB.MAXIMIZE)

    # Step 7: Run Solver and Extract Assignment Map
    model.optimize()

    assignment = {j: [] for j in range(m)}
    if model.status == GRB.OPTIMAL:
        for i in range(n):
            for j in range(m):
                if x[i, j].X > 0.5:  # If the binary variable is active, record the assignment
                    assignment[j].append(i)
    return assignment

def evaluate_metrics(S, assignment):
    """
    Computes performance benchmarks for an evaluation assignment map based on the
    underlying real-world continuous similarity scores.

    Metrics Calculated:
    -------------------
    1. Min Fairness (Max-Min): The absolute lowest cumulative score received by any paper.
                                Measures how well the algorithm protected its worst-off case.
    2. Nash Social Welfare: The geometric mean of all paper scores. Measures global welfare
                            while punishing inequity (a single score close to 0 crashes the geometric mean).
    """
    m = S.shape[1]

    # Calculate the continuous valuation score for each paper by summing its reviewers' similarities
    valuations = [sum(S[i, j] for i in assignment[j]) for j in range(m)]

    # Worst-case fairness metric
    max_min_fairness = min(valuations)

    # Nash Social Welfare metric (Geometric Mean calculation)
    # Uses max(v, 1e-5) to handle edge cases where a paper valuation is 0, avoiding math errors.
    nsw = np.prod([max(v, 1e-5) for v in valuations]) ** (1.0 / m)

    return max_min_fairness, nsw

# ==========================================
# EXECUTION & EXPERIMENTATION BLOCK
# ==========================================
if __name__ == "__main__":
    # --- WORKSPACE CONFIGURATION TOOTLES ---
    N_REVIEWERS = 20        # Size of reviewer pool
    N_PAPERS = 16           # Number of papers submitted
    DEMAND_PER_PAPER = 3    # Reviewers required per paper (lambda)
    LOAD_PER_REVIEWER = 3   # Max reviews per reviewer (mu)

    # Step 1: Generate Dataset
    print(f"Generating matrix for {N_REVIEWERS} Reviewers x {N_PAPERS} Papers...")
    S = generate_dataset(n_reviewers=N_REVIEWERS, n_papers=N_PAPERS,
                         lambda_=DEMAND_PER_PAPER, mu_=LOAD_PER_REVIEWER)

    # Step 2: Run the Fairness-Centric Algorithm (PR4A)
    print("Running PeerReview4All (PR4A)...")
    # PR4A uses a specialized, iterative Max-Flow approach to maximize the minimum paper profile
    pr4a_engine = autoassigner.auto_assigner(S, demand=DEMAND_PER_PAPER, ability=LOAD_PER_REVIEWER)
    pr4a_engine.fair_assignment()
    pr4a_assignment = pr4a_engine.fa

    # Step 3: Run the Efficiency-Centric Algorithm (Binary NSW)
    print("Running Binary NSW Optimization...")
    # Binary NSW maximizes global logarithmic satisfaction based on a similarity threshold
    binary_nsw_assignment = run_binary_nsw(S, lambda_=DEMAND_PER_PAPER, mu_=LOAD_PER_REVIEWER, threshold=0.4)

    # Step 4: Compute Metrics for Comparison
    pr4a_fairness, pr4a_nsw = evaluate_metrics(S, pr4a_assignment)
    bin_fairness, bin_nsw = evaluate_metrics(S, binary_nsw_assignment)

    # Step 5: Format and Print Results Panel
    print("\n" + "="*50)
    print("             EXPERIMENT RESULTS")
    print("="*50)
    print(f"{'Algorithm':<15} | {'Min Fairness (Max-Min)':<22} | {'Nash Social Welfare (NSW)':<25}")
    print("-"*70)
    print(f"{'PR4A':<15} | {pr4a_fairness:<22.4f} | {pr4a_nsw:<25.4f}")
    print(f"{'Binary NSW':<15} | {bin_fairness:<22.4f} | {bin_nsw:<25.4f}")
    print("="*50)

Generating matrix for 20 Reviewers x 16 Papers...
Running PeerReview4All (PR4A)...
Running Binary NSW Optimization...

             EXPERIMENT RESULTS
Algorithm       | Min Fairness (Max-Min) | Nash Social Welfare (NSW)
----------------------------------------------------------------------
PR4A            | 1.9491                 | 2.5419                   
Binary NSW      | 1.6260                 | 1.9610                   


In [ ]:
!git config --global user.email "choudharyrajvil@gmail.com"
!git config --global user.name "RajvilChoudhary"